In [ ]:

import os
import email
import re
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "spamassassin"
FIGURES_DIR = PROJECT_ROOT / "figures"

import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.naive_bayes import ComplementNB


from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    roc_curve, 
    auc,
    accuracy_score,
    precision_recall_fscore_support,
    silhouette_score,
    calinski_harabasz_score
)

import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)


# ==============================================================================
# DATA LOADING AND EXPLORATION
# ==============================================================================

Assembling the dataset begins by walking the five SpamAssassin directories (easy_ham, easy_ham_2, hard_ham, spam, and spam_2). Each file is read with Latin-1 encoding to handle special characters, then parsed with the standard library email module into message objects. Labels are assigned by folder: ham categories map to 0 and spam categories to 1. The loader returns parsed message objects, a label array, and raw text for reference. Dataset size and class distribution are checked in the exploration step to ensure at least 9,260 samples and to document the balance between ham and spam before any modeling.

**`load_emails`** builds the initial dataset from the SpamAssassin directory. It iterates over the five category folders (easy_ham, easy_ham_2, hard_ham, spam, spam_2), reads each file with Latin-1 encoding, and parses it into an email message object. Labels are set by folder: ham categories → 0, spam categories → 1. It returns three aligned outputs: the list of parsed message objects, a numpy array of labels, and the raw file text for reference. Read errors are caught and reported so one bad file does not stop the run.

In [ ]:
def load_emails(base_path=None):
    if base_path is None:
        base_path = DATA_DIR
    base_path = str(base_path)
    """
    Load all emails from the SpamAssassin dataset.
    """
    emails = []
    labels = []
    raw_texts = []
    
    # Define directories and their labels, ham = 0, spam = 1
    email_categories = {
        'easy_ham': 0,
        'easy_ham_2': 0,
        'hard_ham': 0,
        'spam': 1,
        'spam_2': 1
    }
    
    print(f"Loading emails from {base_path}...")
    
    for category, label in email_categories.items():
        category_path = os.path.join(base_path, category)
        
        if not os.path.exists(category_path):
            print(f"Warning: Directory {category_path} not found. Skipping...")
            continue
        
        files = [f for f in os.listdir(category_path) if not f.endswith('.ipynb')]
        print(f"  Loading {len(files)} emails from {category}...")
        
        for filename in files:
            file_path = os.path.join(category_path, filename)
            try:
                with open(file_path, 'r', encoding='latin-1') as file_handler:
                    msg = email.message_from_file(file_handler)
                    emails.append(msg)
                    labels.append(label)
                    file_handler.seek(0)
                    raw_texts.append(file_handler.read())
            except Exception as e:
                print(f"  Error reading {file_path}: {e}")
                continue
    
    print(f"\nTotal emails loaded: {len(emails)}")
    
    return emails, np.array(labels), raw_texts


**`explore_dataset`** summarizes the loaded dataset before modeling. It reports total sample count and class distribution (ham vs spam counts and percentages), checks how many messages have empty payloads, and tallies content types (e.g. text/plain, multipart). It then prints one sample ham and one sample spam email (subject, from, content-type, and a short body preview) so the data structure and balance can be inspected.

In [ ]:
def explore_dataset(emails, labels):

    total_samples = len(emails)
    print(f"\nTotal samples: {total_samples}")
    
    unique, counts = np.unique(labels, return_counts=True)
    class_dist = dict(zip(unique, counts))
    
    print(f"\nClass Distribution:")
    print(f"  Ham (0):  {class_dist.get(0, 0):5d} samples ({class_dist.get(0, 0)/total_samples*100:.2f}%)")
    print(f"  Spam (1): {class_dist.get(1, 0):5d} samples ({class_dist.get(1, 0)/total_samples*100:.2f}%)")
    
    empty_count = 0
    for msg in emails:
        payload = msg.get_payload()
        if isinstance(payload, str) and len(payload.strip()) == 0:
            empty_count += 1
    
    print(f"\nEmpty emails: {empty_count}")
    
    content_types = Counter()
    for msg in emails:
        content_types[msg.get_content_type()] += 1
    
    print(f"\nContent Types:")
    for content_type, count in content_types.most_common(10):
        print(f"  {content_type:30s}: {count:5d} ({count/total_samples*100:.2f}%)")
    
    # Show one ham and one spam example
    ham_idx = np.where(labels == 0)[0][0]
    spam_idx = np.where(labels == 1)[0][0]
    
    print("\n--- SAMPLE HAM EMAIL ---")
    print(f"Subject: {emails[ham_idx].get('Subject', 'No Subject')}")
    print(f"From: {emails[ham_idx].get('From', 'Unknown')}")
    print(f"Content-Type: {emails[ham_idx].get_content_type()}")
    payload = emails[ham_idx].get_payload()
    if isinstance(payload, str):
        print(f"Body preview: {payload[:200]}...")
    else:
        print(f"Body: [Multipart message]")
    
    print("\n--- SAMPLE SPAM EMAIL ---")
    print(f"Subject: {emails[spam_idx].get('Subject', 'No Subject')}")
    print(f"From: {emails[spam_idx].get('From', 'Unknown')}")
    print(f"Content-Type: {emails[spam_idx].get_content_type()}")
    payload = emails[spam_idx].get_payload()
    if isinstance(payload, str):
        print(f"Body preview: {payload[:200]}...")
    else:
        print(f"Body: [Multipart message]")
    
    print("\n" + "="*80)


**`plot_class_and_content_distribution`** builds a single figure with two panels: class distribution (Ham vs Spam counts and percentages) and content-type distribution (top MIME types, e.g. text/plain, text/html). It uses the same counts as the exploration summary so the visualization matches the printed statistics. Useful for reports and to quickly assess balance and content mix.

In [ ]:
def plot_class_and_content_distribution(emails, labels):
    """Class distribution (ham vs spam) and content-type distribution (MIME). Returns fig."""
    unique, counts = np.unique(labels, return_counts=True)
    class_dist = dict(zip(unique, counts))
    ham_count = class_dist.get(0, 0)
    spam_count = class_dist.get(1, 0)
    content_types = Counter()
    for msg in emails:
        content_types[msg.get_content_type()] += 1
    total = len(emails)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Dataset: Class Distribution and Content Types', fontsize=16, fontweight='bold')
    ax1 = axes[0]
    classes = ['Ham', 'Spam']
    values = [ham_count, spam_count]
    colors = ['green', 'red']
    bars = ax1.bar(classes, values, color=colors, alpha=0.7)
    ax1.set_ylabel('Number of emails', fontsize=12)
    ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        pct = 100 * val / total
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2 = axes[1]
    top_n = min(10, len(content_types))
    ct_items = content_types.most_common(top_n)
    ct_labels = [item[0] for item in ct_items]
    ct_counts = [item[1] for item in ct_items]
    y_pos = np.arange(len(ct_labels))
    ax2.barh(y_pos, ct_counts, color='steelblue', alpha=0.8)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(ct_labels, fontsize=9)
    ax2.invert_yaxis()
    ax2.set_xlabel('Number of emails', fontsize=12)
    ax2.set_title('Content Types (MIME)', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    return fig


In [ ]:
class_content_fig = plot_class_and_content_distribution(emails, labels)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
class_content_fig.savefig(str(FIGURES_DIR / "00_class_and_content_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
plt.close(class_content_fig)


In [ ]:
emails, labels, raw_texts = load_emails(DATA_DIR)


In [ ]:
explore_dataset(emails, labels)


# ==============================================================================
# TEXT PREPROCESSING AND CLEANING
# ==============================================================================

Preprocessing turns raw email messages into cleaned text for the Bag-of-Words pipeline. Body text is taken from each message (and optionally the subject) and extracted from both plain and HTML parts, with HTML tags stripped. Text is lowercased; URLs and email addresses are removed; then non-alphanumeric characters are dropped and whitespace is collapsed. Tokenization is implicit via splitting on spaces after filtering. A fixed stopword list is applied and very short tokens are dropped. A simple suffix stemmer runs on the remaining words. Emails that end up empty after cleaning are recorded by index so they can be dropped or handled before feature extraction, keeping the dataset above the required sample count.

**`extract_email_body`** pulls all readable text from a single email message object. For multipart messages it walks each part: plain-text parts are decoded and appended, and HTML parts are decoded, stripped of tags with a regex, then appended. For single-part messages it decodes the payload or falls back to the string form of the payload. Decoding uses UTF-8 with `errors='ignore'` so bad bytes do not raise. The result is one concatenated string (with spaces between parts) for downstream preprocessing.

In [ ]:
def extract_email_body(msg):

    body_text = ""
    
    # Check if message is multipart
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            
            if content_type == "text/plain":
                try:
                    payload = part.get_payload(decode=True)
                    if payload:
                        body_text += payload.decode('utf-8', errors='ignore') + " "
                except:
                    pass
            elif content_type == "text/html":
                try:
                    payload = part.get_payload(decode=True)
                    if payload:
                        html_text = payload.decode('utf-8', errors='ignore')
                        html_text = re.sub(r'<[^>]+>', ' ', html_text)
                        body_text += html_text + " "
                except:
                    pass
    else:
        # Single part message
        try:
            payload = msg.get_payload(decode=True)
            if payload:
                body_text = payload.decode('utf-8', errors='ignore')
            else:
                body_text = str(msg.get_payload())
        except:
            body_text = str(msg.get_payload())
    
    return body_text


**`preprocess_email`** turns one email message into a single cleaned token string for the Bag-of-Words pipeline. It gets the body (and optionally the subject) via `extract_email_body`, strips any remaining HTML, lowercases, then removes URLs and email addresses with regex. It keeps only letters and digits and collapses whitespace. A fixed stopword set is applied and tokens of length 2 or less are dropped. A small inline stemmer strips common suffixes (e.g. *ing*, *ed*, *ly*, *es*, *s*). The result is a space‑joined string of stemmed, non‑stopword tokens ready for vectorization.

In [ ]:
def preprocess_email(msg, include_subject=True):

    text = extract_email_body(msg)
    
    if include_subject:
        subject = msg.get('Subject', '')
        if subject:
            text = subject + " " + text
    
    text = re.sub(r'<[^>]+>', ' ', text)    
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    # Basic stopword removal (common English words that don't add much meaning)
    stopwords = {
        'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
        "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he',
        'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's",
        'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
        'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
        'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
        'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because',
        'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against',
        'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
        'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
        'further', 'then', 'once'
    }
    
    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stopwords and len(word) > 2]
    
    # Simple stemming (remove common suffixes)
    def simple_stem(word):
        """Apply very basic stemming rules."""
        if word.endswith('ing'):
            word = word[:-3]
        elif word.endswith('ed'):
            word = word[:-2]
        elif word.endswith('ly'):
            word = word[:-2]
        elif word.endswith('es'):
            word = word[:-2]
        elif word.endswith('s') and len(word) > 3:
            word = word[:-1]
        return word
    
    words = [simple_stem(word) for word in words]
    text = ' '.join(words)
    
    return text


**`preprocess_all_emails`** runs `preprocess_email` on every message in the list and collects the cleaned strings plus the indices of emails that end up empty. It loops over messages, catches per-email exceptions so one failure does not stop the run (failed or empty messages are recorded in `empty_idices` and get an empty string in `cleaned_emails`), and prints progress every 1,000 emails. It returns `cleaned_emails` and `empty_indices` so the caller can drop empty documents before BOW (e.g. via `remove_empty_emails`).

In [ ]:
def preprocess_all_emails(emails, include_subject=True):

    print(f"Preprocessing {len(emails)} emails...")
    
    cleaned_emails = []
    empty_indices = []
    
    for i, msg in enumerate(emails):
        try:
            cleaned_text = preprocess_email(msg, include_subject=include_subject)
            cleaned_emails.append(cleaned_text)
            if len(cleaned_text.strip()) == 0:
                empty_indices.append(i)
            if (i + 1) % 1000 == 0:
                print(f"  Processed {i + 1}/{len(emails)} emails...")
        except Exception as e:
            print(f"  Error processing email {i}: {e}")
            cleaned_emails.append("")
            empty_indices.append(i)
    
    print(f"  Total emails: {len(cleaned_emails)}")
    print(f"  Empty after cleaning: {len(empty_indices)}")
    
    if len(empty_indices) > 0:
        print(f"  Warning: {len(empty_indices)} emails are empty after preprocessing")
        print(f"  Empty email indices (first 10): {empty_indices[:10]}")
    
    return cleaned_emails, empty_indices


**`display_preprocessing_examples`** prints before/after snippets for a few ham and spam emails so the effect of preprocessing can be checked. For each of `num_examples` ham and spam messages it shows subject, the first 300 characters of the original body (from `extract_email_body`), and the first 300 characters of the cleaned text. That makes it easy to confirm that lowercasing, URL removal, stopwords, and stemming behave as intended without scanning the full corpus.

In [ ]:
def display_preprocessing_examples(emails, labels, cleaned_emails, num_examples=2):

    print("\n" + "="*80)
    print("PREPROCESSING EXAMPLES")
    print("="*80)
    
    # Show ham examples
    ham_indices = np.where(labels == 0)[0]
    print("\n--- HAM EMAIL EXAMPLES ---")
    for i in range(min(num_examples, len(ham_indices))):
        idx = ham_indices[i]
        print(f"\nExample {i+1}:")
        print(f"Subject: {emails[idx].get('Subject', 'No Subject')}")
        
        original_body = extract_email_body(emails[idx])
        print(f"\nOriginal (first 300 chars):")
        print(original_body[:300] + "...")
        
        print(f"\nCleaned (first 300 chars):")
        print(cleaned_emails[idx][:300] + "...")
        print("-" * 80)
    
    # Show spam examples
    spam_indices = np.where(labels == 1)[0]
    print("\n--- SPAM EMAIL EXAMPLES ---")
    for i in range(min(num_examples, len(spam_indices))):
        idx = spam_indices[i]
        print(f"\nExample {i+1}:")
        print(f"Subject: {emails[idx].get('Subject', 'No Subject')}")
        
        original_body = extract_email_body(emails[idx])
        print(f"\nOriginal (first 300 chars):")
        print(original_body[:300] + "...")
        
        print(f"\nCleaned (first 300 chars):")
        print(cleaned_emails[idx][:300] + "...")
        print("-" * 80)


In [ ]:
cleaned_emails, empty_indices = preprocess_all_emails(emails, include_subject=True)


In [ ]:
display_preprocessing_examples(emails, labels, cleaned_emails, num_examples=2)


# ==============================================================================
# BAG-OF-WORDS FEATURE ENGINEERING
# ==============================================================================

The following section converts the cleaned email text into a numerical feature matrix using a Bag-of-Words model with TF-IDF. Emails that are empty after preprocessing are removed so the vectorizer receives only non-empty documents; the pipeline checks that the remaining sample count still meets the minimum requirement. The TF-IDF vectorizer is configured with maximum vocabulary size, minimum and maximum document frequency, and n-gram range (e.g. unigrams and bigrams), fit on the full set of cleaned texts, and used to transform them into a sparse BOW matrix. Phase 3 also reports matrix statistics (shape, sparsity, non-zero count) and sample vocabulary, and runs a feature analysis that summarizes overall and class-specific term importance (total and per-class TF-IDF averages for ham vs spam), with tables and a four-panel figure. Axis and table labels use TF-IDF so the analysis is interpretable.

**`remove_empty_emails`** drops every message whose index is in `empty_indices` from the four aligned lists (emails, cleaned_emails, labels, raw_texts) so the BOW vectorizer never sees empty documents. It builds a mask, filters all four structures, and prints how many were removed and the new size. It then checks whether the remaining sample count is at least 9,260 and prints a warning or confirmation. Returns the four filtered lists for use in the rest of the pipeline.

In [ ]:
def remove_empty_emails(emails, cleaned_emails, labels, raw_texts, empty_indices):
    
    if len(empty_indices) == 0:
        print("\nNo empty emails to remove.")
        return emails, cleaned_emails, labels, raw_texts
    
    print(f"\nRemoving {len(empty_indices)} empty email(s) from dataset...")
    print(f"Empty email indices: {empty_indices}")
    
    mask = np.ones(len(emails), dtype=bool)
    mask[empty_indices] = False

    filtered_emails = [email for i, email in enumerate(emails) if mask[i]]
    filtered_cleaned = [text for i, text in enumerate(cleaned_emails) if mask[i]]
    filtered_labels = labels[mask]
    filtered_raw = [text for i, text in enumerate(raw_texts) if mask[i]]
    
    print(f"Dataset size: {len(emails)} → {len(filtered_emails)}")
    print(f"Removed: {len(emails) - len(filtered_emails)} emails")
    
    if len(filtered_emails) < 9260:
        print(f"WARNING: After removal, dataset has {len(filtered_emails)} samples (< 9260)")
    else:
        print(f"Dataset still meets minimum requirement: {len(filtered_emails)} >= 9260")
    
    return filtered_emails, filtered_cleaned, filtered_labels, filtered_raw


**`create_bow_features`** turns the list of cleaned email strings into a numeric feature matrix using TF-IDF. Parameters include `max_features`, `min_df`, `max_df`, and `ngram_range` (e.g. (1,2) for unigrams and bigrams). The vectorizer is fit on the full corpus and then used to transform it, producing a sparse matrix. The function prints shape, sparsity, non-zero count, and sample vocabulary, and returns the matrix and the fitted TfidfVectorizer for later analysis and modeling.

In [ ]:
def create_bow_features(cleaned_emails, max_features=3000, min_df=2, max_df=0.95, ngram_range=(1, 2)):
    
    print("\n" + "="*80)
    print("CREATING BAG-OF-WORDS FEATURES")
    print("="*80)
    
    print(f"\nVectorizer: TF-IDF")
    print(f"Max features: {max_features}")
    print(f"Min document frequency: {min_df}")
    print(f"Max document frequency: {max_df}")
    print(f"N-gram range: {ngram_range}")
    print("\n  - Weights terms by importance (TF * IDF)")
    print("  - Downweights common terms across documents")
    print("  - Emphasizes distinctive/rare terms")
    
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
        ngram_range=ngram_range,
        lowercase=False,  # Already lowercased in preprocessing
        strip_accents=None,  # Already handled
        stop_words=None  # Already removed stopwords
    )
    
    print("\nFitting vectorizer and transforming emails...")
    bow_matrix = vectorizer.fit_transform(cleaned_emails)
    
    feature_names = vectorizer.get_feature_names_out()
    
    print(f"\nBOW Matrix Statistics:")
    print(f"  Shape: {bow_matrix.shape}")
    print(f"  Number of samples: {bow_matrix.shape[0]}")
    print(f"  Number of features: {bow_matrix.shape[1]}")
    print(f"  Sparsity: {(1.0 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100:.2f}%")
    print(f"  Non-zero entries: {bow_matrix.nnz:,}")
    
    print(f"\nSample features (first 20):")
    print(f"  {list(feature_names[:20])}")
    print(f"\nSample features (last 20):")
    print(f"  {list(feature_names[-20:])}")
    
    return bow_matrix, vectorizer


**`analyze_bow_features`** summarizes term importance for interpretation and sanity checks. It computes per-document sums and class-wise averages (ham vs spam) from the BOW matrix, then prints ranked tables for overall top terms and for terms most associated with ham or spam. It builds a four-panel figure: overall top features, top ham features, top spam features, and a comparison of the most spam-distinctive terms. Axis and table labels use TF-IDF so the values are interpreted correctly.


In [ ]:
def analyze_bow_features(bow_matrix, vectorizer, labels, top_n=20):
    
    print("\n" + "="*80)
    print("ANALYZING BOW FEATURES")
    print("="*80)
    
    freq_label = "TF-IDF"
    print("\n(Values shown are TF-IDF sums/averages, not raw token counts.)")
    
    feature_names = vectorizer.get_feature_names_out()
    
    print("\nCalculating feature statistics...")
    
    overall_freq = np.asarray(bow_matrix.sum(axis=0)).flatten()
    
    ham_mask = labels == 0
    spam_mask = labels == 1
    
    ham_freq = np.asarray(bow_matrix[ham_mask].sum(axis=0)).flatten()
    spam_freq = np.asarray(bow_matrix[spam_mask].sum(axis=0)).flatten()
    
    # Calculate average frequency per document for each class and sort by frequency
    ham_avg = ham_freq / ham_mask.sum()
    spam_avg = spam_freq / spam_mask.sum()
    overall_top_idx = overall_freq.argsort()[-top_n:][::-1]
    ham_top_idx = ham_avg.argsort()[-top_n:][::-1]
    spam_top_idx = spam_avg.argsort()[-top_n:][::-1]
    
    print(f"\n{'='*80}")
    print(f"TOP {top_n} FEATURES OVERALL (by total {freq_label.lower()})")
    print(f"{'='*80}")
    print(f"{'Feature':<30} {'Total':>15} {'Ham':>15} {'Spam':>15}")
    print("-" * 80)
    for idx in overall_top_idx:
        print(f"{feature_names[idx]:<30} {overall_freq[idx]:>15.2f} "
              f"{ham_freq[idx]:>15.2f} {spam_freq[idx]:>15.2f}")
    
    print(f"\n{'='*80}")
    print(f"TOP {top_n} FEATURES IN HAM EMAILS (by average {freq_label.lower()})")
    print(f"{'='*80}")
    print(f"{'Feature':<30} {'Ham Avg':>15} {'Spam Avg':>15} {'Ratio (H/S)':>15}")
    print("-" * 80)
    for idx in ham_top_idx:
        ratio = ham_avg[idx] / (spam_avg[idx] + 1e-10)  # Add small value to avoid division by zero
        print(f"{feature_names[idx]:<30} {ham_avg[idx]:>15.4f} "
              f"{spam_avg[idx]:>15.4f} {ratio:>15.2f}")
    
    print(f"\n{'='*80}")
    print(f"TOP {top_n} FEATURES IN SPAM EMAILS (by average {freq_label.lower()})")
    print(f"{'='*80}")
    print(f"{'Feature':<30} {'Spam Avg':>15} {'Ham Avg':>15} {'Ratio (S/H)':>15}")
    print("-" * 80)
    for idx in spam_top_idx:
        ratio = spam_avg[idx] / (ham_avg[idx] + 1e-10)
        print(f"{feature_names[idx]:<30} {spam_avg[idx]:>15.4f} "
              f"{ham_avg[idx]:>15.4f} {ratio:>15.2f}")
    
    # Set up the plot
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Bag-of-Words Feature Analysis', fontsize=20, fontweight='bold')
    
    # Plot 1: Overall top features
    ax1 = axes[0, 0]
    top_features = feature_names[overall_top_idx]
    top_freqs = overall_freq[overall_top_idx]
    ax1.barh(range(len(top_features)), top_freqs, color='steelblue')
    ax1.set_yticks(range(len(top_features)))
    ax1.set_yticklabels(top_features, fontsize=10)
    ax1.invert_yaxis()
    ax1.set_xlabel(f'Total {freq_label}', fontsize=12, fontweight='bold')
    ax1.set_title(f'Top {top_n} Features Overall', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # Plot 2: Ham-specific features
    ax2 = axes[0, 1]
    ham_features = feature_names[ham_top_idx]
    ham_freqs = ham_avg[ham_top_idx]
    ax2.barh(range(len(ham_features)), ham_freqs, color='green', alpha=0.7)
    ax2.set_yticks(range(len(ham_features)))
    ax2.set_yticklabels(ham_features, fontsize=10)
    ax2.invert_yaxis()
    ax2.set_xlabel(f'Average {freq_label} in Ham', fontsize=12, fontweight='bold')
    ax2.set_title(f'Top {top_n} Ham Features', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # Plot 3: Spam-specific features
    ax3 = axes[1, 0]
    spam_features = feature_names[spam_top_idx]
    spam_freqs = spam_avg[spam_top_idx]
    ax3.barh(range(len(spam_features)), spam_freqs, color='red', alpha=0.7)
    ax3.set_yticks(range(len(spam_features)))
    ax3.set_yticklabels(spam_features, fontsize=10)
    ax3.invert_yaxis()
    ax3.set_xlabel(f'Average {freq_label} in Spam', fontsize=12, fontweight='bold')
    ax3.set_title(f'Top {top_n} Spam Features', fontsize=14, fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
    
    # Plot 4: Comparison of Ham vs Spam for distinctive features
    ax4 = axes[1, 1]
    # Get features that are distinctive (high in one class, low in other)
    spam_ratio = spam_avg / (ham_avg + 1e-10)
    spam_distinctive_idx = spam_ratio.argsort()[-top_n:][::-1]
    
    x_pos = np.arange(top_n)
    width = 0.35
    
    distinctive_features = feature_names[spam_distinctive_idx]
    distinctive_ham = ham_avg[spam_distinctive_idx]
    distinctive_spam = spam_avg[spam_distinctive_idx]
    
    ax4.bar(x_pos - width/2, distinctive_ham, width, label='Ham', color='green', alpha=0.7)
    ax4.bar(x_pos + width/2, distinctive_spam, width, label='Spam', color='red', alpha=0.7)
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(distinctive_features, rotation=45, ha='right', fontsize=9)
    ax4.set_ylabel(f'Average {freq_label}', fontsize=12, fontweight='bold')
    ax4.set_title('Most Distinctive Spam Features (Ham vs Spam)', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    print("\nFeature analysis complete!")
    print("Visualization created (4 subplots showing feature distributions)")
    
    return fig


In [ ]:
# Phase 3: BOW Features
bow_matrix, vectorizer = create_bow_features(
    cleaned_emails,
    max_features=3000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)


In [ ]:
# Analyze features to understand spam vs ham patterns
feature_analysis_fig = analyze_bow_features(bow_matrix, vectorizer, labels, top_n=20)


# ==============================================================================
# CLUSTERING FOR FEATURE ENGINEERING
# ==============================================================================

I'm using unsupervised clustering on the BOW feature matrix to create additional inputs for the classifier. The pipeline first determines a reasonable number of clusters by evaluating k in a range (e.g. 2 to 30) on a subsample of the data for speed. Three metrics are computed for each k: inertia (elbow method), silhouette score, and Calinski-Harabasz index; results are plotted in a four-panel figure (elbow, silhouette, CH, and a normalized comparison). The chosen k is justified from these metrics (e.g. best silhouette or a compromise with the elbow). A single clustering model is then fit on the full BOW matrix and cluster labels are assigned to every email. Cluster composition is analyzed by spam/ham counts and percentages per cluster, with tables and figures (cluster sizes, spam percentage by cluster, stacked ham/spam, and a classification summary). Cluster assignments are one-hot encoded and concatenated with the BOW matrix so the Naive Bayes model receives both term features and cluster-membership features.

**`determine_optimal_clusters`** evaluates a range of k values to choose a cluster count. It can subsample the BOW matrix for speed (with a fixed seed). For each k it fits K-Means (or MiniBatch K-Means on large data), then computes inertia, silhouette score, and Calinski–Harabasz index. It prints a table of these metrics and builds a four-panel figure: elbow (inertia), silhouette, Calinski–Harabasz, and a normalized comparison. It returns a dict of the metrics and the figure, and prints recommendations (e.g. best k by silhouette and by CH) so k can be chosen with justification.

In [ ]:
def determine_optimal_clusters(bow_matrix, k_range=range(2, 31), sample_size=None):
    
    print("\n" + "="*80)
    print("DETERMINING OPTIMAL NUMBER OF CLUSTERS")
    print("="*80)
    
    print(f"\nTesting k values from {min(k_range)} to {max(k_range)}")
    print(f"Original matrix shape: {bow_matrix.shape}")
    
    if sample_size and bow_matrix.shape[0] > sample_size:
        print(f"\nSampling {sample_size} points for faster computation...")
        np.random.seed(42)
        sample_indices = np.random.choice(bow_matrix.shape[0], sample_size, replace=False)
        X_sample = bow_matrix[sample_indices]
        print(f"Sample matrix shape: {X_sample.shape}")
    else:
        X_sample = bow_matrix
        print(f"Using full dataset: {X_sample.shape}")
    
    inertias = []
    silhouette_scores = []
    calinski_harabasz_scores = []
    k_values = list(k_range)
    
    print("\nEvaluating clusters...")
    print(f"{'K':>5} {'Inertia':>15} {'Silhouette':>15} {'Calinski-H':>15} {'Time (s)':>10}")
    print("-" * 70)
    
    for k in k_values:
        start_time = time.time()
        
        # Fit K-Means
        if X_sample.shape[0] > 5000:
            kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1000, n_init=3)
        else:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        
        cluster_labels = kmeans.fit_predict(X_sample)
        inertia = kmeans.inertia_
        silhouette = silhouette_score(X_sample, cluster_labels, sample_size=min(5000, X_sample.shape[0]))
        calinski = calinski_harabasz_score(X_sample.toarray() if hasattr(X_sample, 'toarray') else X_sample, cluster_labels)
        
        inertias.append(inertia)
        silhouette_scores.append(silhouette)
        calinski_harabasz_scores.append(calinski)
        
        elapsed = time.time() - start_time
        print(f"{k:>5} {inertia:>15.2f} {silhouette:>15.4f} {calinski:>15.2f} {elapsed:>10.2f}")
    
    # Create visualization of all metrics
    print("\nCreating cluster evaluation visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Cluster Number Optimization Analysis', fontsize=20, fontweight='bold')
    
    # Plot 1: Elbow Method (Inertia)
    ax1 = axes[0, 0]
    ax1.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12, fontweight='bold')
    ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(k_values[::2])  # Show every other k value
    
    # Add annotation for potential elbow
    # Calculate rate of change to identify elbow
    if len(inertias) > 2:
        diffs = np.diff(inertias)
        second_diffs = np.diff(diffs)
        elbow_idx = np.argmax(second_diffs) + 1
        if elbow_idx < len(k_values):
            ax1.axvline(x=k_values[elbow_idx], color='red', linestyle='--', alpha=0.7, 
                       label=f'Potential elbow at k={k_values[elbow_idx]}')
            ax1.legend()
    
    # Plot 2: Silhouette Score
    ax2 = axes[0, 1]
    ax2.plot(k_values, silhouette_scores, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
    ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(k_values[::2])
    
    # Add horizontal lines for interpretation
    ax2.axhline(y=0.5, color='darkgreen', linestyle=':', alpha=0.5, label='Good (>0.5)')
    ax2.axhline(y=0.3, color='orange', linestyle=':', alpha=0.5, label='Reasonable (>0.3)')
    
    # Mark best silhouette score
    best_sil_idx = np.argmax(silhouette_scores)
    ax2.plot(k_values[best_sil_idx], silhouette_scores[best_sil_idx], 'r*', 
            markersize=20, label=f'Best: k={k_values[best_sil_idx]}')
    ax2.legend()
    
    # Plot 3: Calinski-Harabasz Index
    ax3 = axes[1, 0]
    ax3.plot(k_values, calinski_harabasz_scores, 'mo-', linewidth=2, markersize=8)
    ax3.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Calinski-Harabasz Index', fontsize=12, fontweight='bold')
    ax3.set_title('Calinski-Harabasz Analysis', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_xticks(k_values[::2])
    
    # Mark best CH score
    best_ch_idx = np.argmax(calinski_harabasz_scores)
    ax3.plot(k_values[best_ch_idx], calinski_harabasz_scores[best_ch_idx], 'r*', 
            markersize=20, label=f'Best: k={k_values[best_ch_idx]}')
    ax3.legend()
    
    # Plot 4: Combined normalized comparison
    ax4 = axes[1, 1]
    
    # Normalize all metrics to 0-1 scale for comparison
    norm_inertia = 1 - (np.array(inertias) - min(inertias)) / (max(inertias) - min(inertias))  # Invert: lower is better
    norm_silhouette = (np.array(silhouette_scores) - min(silhouette_scores)) / \
                     (max(silhouette_scores) - min(silhouette_scores))
    norm_ch = (np.array(calinski_harabasz_scores) - min(calinski_harabasz_scores)) / \
             (max(calinski_harabasz_scores) - min(calinski_harabasz_scores))
    
    ax4.plot(k_values, norm_inertia, 'b-', linewidth=2, marker='o', label='Elbow (inverted)', alpha=0.7)
    ax4.plot(k_values, norm_silhouette, 'g-', linewidth=2, marker='s', label='Silhouette', alpha=0.7)
    ax4.plot(k_values, norm_ch, 'm-', linewidth=2, marker='^', label='Calinski-H', alpha=0.7)
    
    ax4.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Normalized Score (0-1)', fontsize=12, fontweight='bold')
    ax4.set_title('Normalized Comparison of All Metrics', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend(fontsize=11)
    ax4.set_xticks(k_values[::2])
    
    plt.tight_layout()
    
    # Print recommendations
    print("\n" + "="*80)
    print("CLUSTER NUMBER RECOMMENDATIONS")
    print("="*80)
    
    best_sil_k = k_values[best_sil_idx]
    best_ch_k = k_values[best_ch_idx]
    
    print(f"\nBest by Silhouette Score: k={best_sil_k} (score={silhouette_scores[best_sil_idx]:.4f})")
    print(f"Best by Calinski-Harabasz: k={best_ch_k} (score={calinski_harabasz_scores[best_ch_idx]:.2f})")
    
    # Suggest reasonable range
    good_sil_k = [k for k, s in zip(k_values, silhouette_scores) if s > 0.3]
    if good_sil_k:
        print(f"\nClusters with good silhouette (>0.3): {good_sil_k}")
    
    print("\nGuidance for selection:")
    print("  - Silhouette score prioritizes well-separated clusters")
    print("  - Calinski-Harabasz prioritizes dense, distinct clusters")
    print("  - Elbow method suggests diminishing returns point")
    print("  - Consider computational cost vs interpretability")
    print("  - For this task: 10-20 clusters often works well for text data")
    
    metrics_dict = {
        'k_values': k_values,
        'inertias': inertias,
        'silhouette_scores': silhouette_scores,
        'calinski_harabasz_scores': calinski_harabasz_scores,
        'best_silhouette_k': best_sil_k,
        'best_ch_k': best_ch_k
    }
    
    return metrics_dict, fig


**`fit_clustering_model`** fits Mini-Batch K-Means on the full BOW matrix with the chosen number of clusters. It returns the fitted model and the cluster label for every sample. It prints runtime, inertia, iteration count, and the size distribution of the clusters (counts and percentages). These labels are later one-hot encoded and appended to the BOW features for the classifier.

In [ ]:
def fit_clustering_model(bow_matrix, n_clusters):

    print("\n" + "="*80)
    print("FITTING CLUSTERING MODEL")
    print("="*80)
    
    print(f"\nNumber of clusters: {n_clusters}")
    print(f"Dataset shape: {bow_matrix.shape}")
    print("\nUsing Mini-Batch K-Means")
    
    model = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=42,
        batch_size=1000,
        n_init=10,
        max_iter=100,
        verbose=0
    )
    
    print("Fitting model...")
    start_time = time.time()
    cluster_labels = model.fit_predict(bow_matrix)
    elapsed = time.time() - start_time
    
    print(f"Clustering complete in {elapsed:.2f} seconds")
    print(f"\nCluster Statistics:")
    print(f"  Inertia: {model.inertia_:.2f}")
    print(f"  Number of iterations: {model.n_iter_}")
    
    # Calculate cluster sizes
    unique_clusters, cluster_counts = np.unique(cluster_labels, return_counts=True)
    print(f"\nCluster size distribution:")
    print(f"  {'Cluster':>10} {'Size':>10} {'Percentage':>12}")
    print("-" * 35)
    for cluster_id, count in zip(unique_clusters, cluster_counts):
        percentage = (count / len(cluster_labels)) * 100
        print(f"  {cluster_id:>10} {count:>10} {percentage:>11.2f}%")
    
    print(f"\nCluster size statistics:")
    print(f"  Mean: {np.mean(cluster_counts):.1f}")
    print(f"  Std:  {np.std(cluster_counts):.1f}")
    print(f"  Min:  {np.min(cluster_counts)}")
    print(f"  Max:  {np.max(cluster_counts)}")
    
    return model, cluster_labels


**`analyze_clusters`** describes what the clusters capture. It computes per-cluster counts and spam percentage using the true labels (for interpretation only; clustering stays unsupervised). It assigns a label such as VERY SPAM or HAM-HEAVY, prints a table and the top terms per cluster (from mean BOW/TF-IDF in that cluster), and builds a four-panel figure: cluster sizes, spam percentage by cluster, stacked ham/spam counts, and a pie chart of cluster classifications. Colors use a fixed order (e.g. spam=red, ham=green) for consistency.

In [ ]:
def analyze_clusters(cluster_labels, labels, cleaned_emails, vectorizer, bow_matrix):

    print("\n" + "="*80)
    print("ANALYZING CLUSTERS")
    print("="*80)
    
    n_clusters = len(np.unique(cluster_labels))
    
    # Calculate spam/ham distribution per cluster
    print(f"\nSpam/Ham Distribution by Cluster:")
    print(f"{'Cluster':>10} {'Total':>8} {'Ham':>8} {'Spam':>8} {'% Spam':>10} {'Classification':>15}")
    print("-" * 75)
    
    cluster_info = []
    for cluster_id in range(n_clusters):
        mask = cluster_labels == cluster_id
        cluster_size = mask.sum()
        
        ham_count = ((labels == 0) & mask).sum()
        spam_count = ((labels == 1) & mask).sum()
        spam_pct = (spam_count / cluster_size) * 100 if cluster_size > 0 else 0
        
        classification = "SPAM-HEAVY" if spam_pct > 50 else "HAM-HEAVY"
        if spam_pct > 75:
            classification = "VERY SPAM"
        elif spam_pct < 25:
            classification = "VERY HAM"
        
        cluster_info.append({
            'cluster_id': cluster_id,
            'total': cluster_size,
            'ham': ham_count,
            'spam': spam_count,
            'spam_pct': spam_pct,
            'classification': classification
        })
        
        print(f"{cluster_id:>10} {cluster_size:>8} {ham_count:>8} {spam_count:>8} "
              f"{spam_pct:>9.1f}% {classification:>15}")
    
    print(f"\n{'='*80}")
    print("TOP TERMS PER CLUSTER")
    print(f"{'='*80}")
    
    feature_names = vectorizer.get_feature_names_out()
    
    for cluster_id in range(min(10, n_clusters)):  # Show first 10 clusters
        mask = cluster_labels == cluster_id
        cluster_bow = bow_matrix[mask]
        mean_tfidf = np.asarray(cluster_bow.mean(axis=0)).flatten()
        top_indices = mean_tfidf.argsort()[-10:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        info = cluster_info[cluster_id]

        print(f"\nCluster {cluster_id} ({info['classification']}, {info['spam_pct']:.1f}% spam):")
        print(f"  Top terms: {', '.join(top_terms)}")
    
    if n_clusters > 10:
        print(f"\n... (showing first 10 of {n_clusters} clusters)")
    
    # Create visualizations
    print("\nCreating cluster analysis visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Cluster Analysis', fontsize=20, fontweight='bold')
    
    # Plot 1: Cluster sizes
    ax1 = axes[0, 0]
    cluster_ids = [info['cluster_id'] for info in cluster_info]
    cluster_sizes = [info['total'] for info in cluster_info]
    ax1.bar(cluster_ids, cluster_sizes, color='steelblue', alpha=0.7)
    ax1.set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Emails', fontsize=12, fontweight='bold')
    ax1.set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot 2: Spam percentage by cluster
    ax2 = axes[0, 1]
    spam_pcts = [info['spam_pct'] for info in cluster_info]
    colors = ['red' if pct > 50 else 'green' for pct in spam_pcts]
    ax2.bar(cluster_ids, spam_pcts, color=colors, alpha=0.7)
    ax2.axhline(y=50, color='black', linestyle='--', alpha=0.5, label='50% threshold')
    ax2.set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Spam Percentage', fontsize=12, fontweight='bold')
    ax2.set_title('Spam Percentage by Cluster', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Plot 3: Stacked bar chart of ham vs spam
    ax3 = axes[1, 0]
    ham_counts = [info['ham'] for info in cluster_info]
    spam_counts = [info['spam'] for info in cluster_info]
    ax3.bar(cluster_ids, ham_counts, label='Ham', color='green', alpha=0.7)
    ax3.bar(cluster_ids, spam_counts, bottom=ham_counts, label='Spam', color='red', alpha=0.7)
    ax3.set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Number of Emails', fontsize=12, fontweight='bold')
    ax3.set_title('Ham vs Spam Distribution by Cluster', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Plot 4: Cluster classification summary
    # Use fixed order so pie colors match semantics (spam=red, ham=green)
    ax4 = axes[1, 1]
    classifications = [info['classification'] for info in cluster_info]
    class_counts = Counter(classifications)
    order = ['VERY SPAM', 'SPAM-HEAVY', 'HAM-HEAVY', 'VERY HAM']
    pie_labels = [k for k in order if k in class_counts]
    pie_sizes = [class_counts[k] for k in pie_labels]
    pie_colors = ['darkred', 'red', 'lightgreen', 'darkgreen'][:len(pie_labels)]
    ax4.pie(pie_sizes, labels=pie_labels, colors=pie_colors, autopct='%1.1f%%',
           startangle=90)
    ax4.set_title('Cluster Classification Distribution', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    print("\nCluster analysis complete!")
    
    return cluster_info, fig


**`add_cluster_features`** concatenates cluster membership with the BOW matrix for modeling. It one-hot encodes the cluster labels (one binary column per cluster) and horizontally stacks that matrix with the BOW matrix. The result is a sparse matrix with BOW features plus cluster indicators. It prints the combined shape and sparsity. The classifier is then trained on this combined matrix so it can use both term weights and cluster assignment.

In [ ]:
def add_cluster_features(bow_matrix, cluster_labels):

    print("\n" + "="*80)
    print("ADDING CLUSTER FEATURES")
    print("="*80)
    
    print(f"\nOriginal BOW matrix shape: {bow_matrix.shape}")
    print(f"Number of clusters: {len(np.unique(cluster_labels))}")
    
    n_clusters = len(np.unique(cluster_labels))
    n_samples = len(cluster_labels)
    
    cluster_features = np.zeros((n_samples, n_clusters))
    cluster_features[np.arange(n_samples), cluster_labels] = 1
    cluster_features_sparse = csr_matrix(cluster_features)
    
    print(f"Cluster features shape: {cluster_features_sparse.shape}")
    
    combined_matrix = hstack([bow_matrix, cluster_features_sparse])
    
    print(f"\nCombined matrix shape: {combined_matrix.shape}")
    print(f"  BOW features: {bow_matrix.shape[1]}")
    print(f"  Cluster features: {n_clusters}")
    print(f"  Total features: {combined_matrix.shape[1]}")
    
    print(f"\nMatrix sparsity: {(1.0 - combined_matrix.nnz / (combined_matrix.shape[0] * combined_matrix.shape[1])) * 100:.2f}%")
    
    print("\nCluster features added successfully!")
    print("Each cluster is now a binary feature (0 or 1) added to the BOW matrix.")
    
    return combined_matrix


In [ ]:
# Determine optimal number of clusters
# Test k from 2 to 30, sample 3000 points for speed
cluster_metrics, cluster_eval_fig = determine_optimal_clusters(
    bow_matrix, 
    k_range=range(2, 31),
    sample_size=3000
)


In [ ]:
# Select number of clusters based on analysis
# Using the best silhouette score as primary criterion
optimal_k = cluster_metrics['best_silhouette_k']
print(f"SELECTED NUMBER OF CLUSTERS: {optimal_k}")
print(f"  - Silhouette score: {cluster_metrics['silhouette_scores'][cluster_metrics['k_values'].index(optimal_k)]:.4f}")


In [ ]:
# Fit final clustering model with chosen k
clustering_model, cluster_labels = fit_clustering_model(
    bow_matrix, 
    n_clusters=optimal_k
)


In [ ]:
# Analyze clusters to understand what they represent
cluster_info, cluster_analysis_fig = analyze_clusters(
    cluster_labels, 
    labels, 
    cleaned_emails, 
    vectorizer, 
    bow_matrix
)


In [ ]:
# Add cluster assignments as new features
combined_matrix = add_cluster_features(bow_matrix, cluster_labels)


# ==============================================================================
# NAIVE BAYES CLASSIFIER
# ==============================================================================

Phase 5 builds and evaluates a single Naive Bayes classifier on the combined feature matrix (BOW plus cluster one-hot features). The classifier type is chosen in advance and documented with a justification; it is not tuned as a hyperparameter (per rubric). The implementation uses ComplementNB by default, justified by the imbalanced class distribution (e.g. 74% ham, 26% spam), which ComplementNB is designed to handle. Training and evaluation use only stratified k-fold cross-validation: no train/test split. StratifiedKFold preserves class proportions in each fold. For every sample, the pipeline obtains an out-of-fold prediction and, when available, prediction probabilities via cross_val_predict, so the confusion matrix and all metrics are based on 100% of the data with each sample predicted only when held out. Per-fold accuracies are computed for reporting, and overall accuracy is reported across all out-of-fold predictions. Probabilities are retained for Phase 6 threshold analysis.

**`train_with_cross_validation`** runs stratified k-fold CV on the feature matrix and labels: no train/test split. It uses StratifiedKFold so each fold keeps roughly the same class balance. For every sample it gets an out-of-fold predicted label and predicted class probabilities via `cross_val_predict`. It also iterates over the same splits to compute per-fold accuracy and prints mean accuracy, min/max fold accuracy, and overall accuracy from the out-of-fold predictions. Returns the out-of-fold predictions, probabilities, per-fold accuracies, and the CV object for downstream confusion matrix and threshold analysis.

In [ ]:
def train_with_cross_validation(X, y, classifier, n_folds=10):

    print(f"\nCross-validation setup:")
    print(f"  Strategy: Stratified K-Fold")
    print(f"  Number of folds: {n_folds}")
    print(f"  Classifier: {classifier.__class__.__name__}")
    
    # Verify dataset size
    print(f"\nDataset statistics:")
    print(f"  Total samples: {len(y)}")
    print(f"  Features: {X.shape[1]}")
    unique, counts = np.unique(y, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = "Ham" if label == 0 else "Spam"
        print(f"  {label_name} ({label}): {count} ({count/len(y)*100:.2f}%)")
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    print(f"\nEach fold will have approximately:")
    print(f"  Training samples: {len(y) * (n_folds-1) / n_folds:.0f}")
    print(f"  Testing samples: {len(y) / n_folds:.0f}")
    
    # Get out-of-fold predictions for all samples
    print("\nTraining and predicting...")
    print("(Each sample is predicted when it's in the test set, never trained on)")
    
    start_time = time.time()
    
    y_pred = cross_val_predict(classifier, X, y, cv=cv, method='predict')
    y_pred_proba = cross_val_predict(classifier, X, y, cv=cv, method='predict_proba')
    elapsed = time.time() - start_time

    print(f"Cross-validation complete in {elapsed:.2f} seconds")
    print("\nCalculating per-fold accuracy...")

    fold_scores = []
    
    fold_num = 1
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        classifier.fit(X_train, y_train)

        y_fold_pred = classifier.predict(X_test)
        
        fold_acc = accuracy_score(y_test, y_fold_pred)
        fold_scores.append(fold_acc)
        
        print(f"  Fold {fold_num:2d}: Accuracy = {fold_acc:.4f} "f"(Train: {len(train_idx)}, Test: {len(test_idx)})")
        fold_num += 1
    
    mean_acc = np.mean(fold_scores)
    std_acc = np.std(fold_scores)
    
    print(f"\n{'='*80}")
    print("CROSS-VALIDATION RESULTS")
    print(f"{'='*80}")
    print(f"Mean Accuracy: {mean_acc:.4f} (±{std_acc:.4f})")
    print(f"Min Accuracy:  {np.min(fold_scores):.4f}")
    print(f"Max Accuracy:  {np.max(fold_scores):.4f}")
    
    # Overall accuracy from out-of-fold predictions
    overall_acc = accuracy_score(y, y_pred)
    print(f"\nOverall Accuracy (all out-of-fold predictions): {overall_acc:.4f}")
    
    return y_pred, y_pred_proba, fold_scores, cv


In [ ]:
# Add cluster assignments as new features
combined_matrix = add_cluster_features(bow_matrix, cluster_labels)


Four Naive Bayes variants were considered for spam classification. **MultinomialNB** models word counts with a multinomial distribution and is a standard for text; it can favor the majority class when the dataset is imbalanced (here, about 74% ham and 26% spam). **BernoulliNB** treats features as binary (present/absent), which discards TF-IDF magnitude and is less natural for our continuous features. **GaussianNB** assumes normally distributed features and is poorly suited to sparse, non-negative TF-IDF matrices. **ComplementNB** was chosen as the classifier: it is intended for imbalanced text, estimates parameters from the complement of each class to reduce bias toward the majority, and is well aligned with TF-IDF. With Laplace smoothing (alpha=1.0), it handles unseen terms and is the default selection for this pipeline.

In [ ]:
# ComplementNB for imbalanced text
classifier = ComplementNB(alpha=1.0)


In [ ]:
# Train using stratified cross-validation
# Using 10 folds for robust evaluation
# StratifiedKFold maintains class proportions in each fold
y_pred, y_pred_proba, fold_scores, cv_object = train_with_cross_validation(
    X=combined_matrix,
    y=labels,
    classifier=classifier,
    n_folds=10
)


# ==============================================================================
# RESULTS ANALYSIS
# ==============================================================================

Phase 6 analyzes model performance using the out-of-fold predictions and probabilities from Phase 5. All metrics and the confusion matrix are computed on 100% of the data, with each sample counted only when it was in the test fold. The pipeline produces a labeled confusion matrix (counts and percentages, TN/FP/FN/TP), a classification report (precision, recall, F1, support per class and macro/weighted), and a misclassification analysis that identifies false positives and false negatives, shows example emails, and discusses possible reasons and improvement paths while staying within the rubric (e.g. no alternative classifiers). Threshold analysis shows how changing the decision threshold affects precision, recall, and accuracy (with plots and ROC curve plus AUC), and summarizes trade-offs (lower vs higher threshold). Additional visualizations include per-fold CV accuracy and per-class metrics. Final accuracy across all folds and overall accuracy from out-of-fold predictions are reported. Figures are sized and labeled for presentation use.

**`create_confusion_matrix_plot`** builds the confusion matrix from true labels and out-of-fold predicted labels (discrete predictions, not probabilities). It prints TN, FP, FN, TP and their percentages, then draws a single heatmap with counts and percentages in each cell, TN/FP/FN/TP labels in the corners, and clear axis labels (Ham/Spam). The title includes total sample count and accuracy. Returns the figure and the confusion matrix array for downstream use.

In [ ]:
def create_confusion_matrix_plot(y_true, y_pred):    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Extract values
    tn, fp, fn, tp = cm.ravel()
    total = tn + fp + fn + tp
    
    print(f"\nConfusion Matrix Values:")
    print(f"  True Negatives (TN):  {tn:5d} (Ham correctly classified as Ham)")
    print(f"  False Positives (FP): {fp:5d} (Ham incorrectly classified as Spam)")
    print(f"  False Negatives (FN): {fn:5d} (Spam incorrectly classified as Ham)")
    print(f"  True Positives (TP):  {tp:5d} (Spam correctly classified as Spam)")
    print(f"  Total:                {total:5d}")
    
    # Calculate percentages
    print(f"\nPercentages:")
    print(f"  True Negatives:  {tn/total*100:6.2f}%")
    print(f"  False Positives: {fp/total*100:6.2f}%")
    print(f"  False Negatives: {fn/total*100:6.2f}%")
    print(f"  True Positives:  {tp/total*100:6.2f}%")
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Create heatmap with both counts and percentages
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                cbar_kws={'label': 'Count'}, ax=ax,
                linewidths=2, linecolor='black')
    
    # Add custom annotations with both count and percentage
    # Using data coordinates for proper positioning within heatmap cells
    for i in range(2):
        for j in range(2):
            count = cm[i, j]
            percentage = (count / total) * 100
            
            # Main count and percentage (centered in cell)
            text = f'{count:,}\n({percentage:.2f}%)'
            ax.text(j + 0.5, i + 0.5, text,
                   ha='center', va='center',
                   fontsize=18, fontweight='bold',
                   color='white' if count > total/4 else 'black')
            
            # Add small TN/FP/FN/TP labels in corner of each cell
            # Using data coordinates for correct positioning
            label_map = {(0, 0): 'TN', (0, 1): 'FP', (1, 0): 'FN', (1, 1): 'TP'}
            cell_label = label_map.get((i, j), '')
            ax.text(j + 0.15, i + 0.15, cell_label,
                   ha='left', va='top',
                   fontsize=11, style='italic', fontweight='bold',
                   color='lightgray' if count > total/4 else 'gray')
    
    # Set title and labels (standard matplotlib approach)
    ax.set_title('Confusion Matrix: Spam Classification Results\n' + 
                 f'Total Samples: {total:,} | Accuracy: {(tn+tp)/total*100:.2f}%',
                 fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Label', fontsize=16, fontweight='bold', labelpad=10)
    ax.set_ylabel('True Label', fontsize=16, fontweight='bold', labelpad=10)
    
    # Set tick labels with proper formatting
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Ham (0)', 'Spam (1)'], fontsize=14, fontweight='bold')
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Ham (0)', 'Spam (1)'], fontsize=14, fontweight='bold', rotation=0)
    
    # Adjust layout with extra padding for labels
    plt.tight_layout(pad=2.0)
    
    return fig, cm


**`generate_classification_metrics`** computes and prints the full classification report (precision, recall, F1, support per class) and a detailed breakdown: macro and weighted averages, specificity, sensitivity, and false positive/negative rates. It takes true labels and out-of-fold predictions. All values are stored in a dictionary (accuracy, per-class metrics, TN/FP/FN/TP, etc.) for the performance summary figure and final reporting.

In [ ]:
def generate_classification_metrics(y_true, y_pred):    
    # Generate classification report
    print("\nClassification Report:")
    print("-" * 80)
    report = classification_report(y_true, y_pred, target_names=['Ham (0)', 'Spam (1)'], digits=4)
    print(report)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # True Negative Rate
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # True Positive Rate (Recall for Spam)
    
    print("\n" + "="*80)
    print("DETAILED METRICS BREAKDOWN")
    print("="*80)
    
    print("\nOverall Metrics:")
    print(f"  Accuracy:  {accuracy:.4f}")
    
    print("\nPer-Class Metrics:")
    print(f"  {'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-" * 70)
    print(f"  {'Ham (0)':<15} {precision[0]:<12.4f} {recall[0]:<12.4f} {f1[0]:<12.4f} {support[0]:<10.0f}")
    print(f"  {'Spam (1)':<15} {precision[1]:<12.4f} {recall[1]:<12.4f} {f1[1]:<12.4f} {support[1]:<10.0f}")
    
    print("\nAverage Metrics:")
    print(f"  Macro Average:    Precision={macro_precision:.4f}, Recall={macro_recall:.4f}, F1={macro_f1:.4f}")
    print(f"  Weighted Average: Precision={weighted_precision:.4f}, Recall={weighted_recall:.4f}, F1={weighted_f1:.4f}")
    
    print("\nAdditional Metrics:")
    print(f"  Specificity (TNR): {specificity:.4f} (Ham correctly identified)")
    print(f"  Sensitivity (TPR): {sensitivity:.4f} (Spam correctly identified)")
    print(f"  False Positive Rate: {fp/(fp+tn):.4f} (Ham misclassified as Spam)")
    print(f"  False Negative Rate: {fn/(fn+tp):.4f} (Spam misclassified as Ham)")
    
    # Store all metrics
    metrics_dict = {
        'accuracy': accuracy,
        'ham_precision': precision[0],
        'ham_recall': recall[0],
        'ham_f1': f1[0],
        'spam_precision': precision[1],
        'spam_recall': recall[1],
        'spam_f1': f1[1],
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_precision,
        'weighted_recall': weighted_recall,
        'weighted_f1': weighted_f1,
        'specificity': specificity,
        'sensitivity': sensitivity,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp
    }
    
    return metrics_dict


**`analyze_misclassifications`** finds false positives (ham predicted as spam) and false negatives (spam predicted as ham) from the out-of-fold predictions. It prints counts and percentages, then shows up to `num_examples` example emails per type with a short preview and possible reasons (e.g. spam-like words in ham, short or newsletter-like spam). It ends with improvement suggestions (e.g. threshold, feature engineering) and a reminder to stay with Naive Bayes per the rubric. Returns the FP and FN index arrays for further inspection.

In [ ]:
def analyze_misclassifications(y_true, y_pred, cleaned_emails, labels, num_examples=10):
    
    misclassified = y_true != y_pred
    
    # False Positives: Ham predicted as Spam
    fp_mask = (y_true == 0) & (y_pred == 1)
    fp_indices = np.where(fp_mask)[0]
    
    # False Negatives: Spam predicted as Ham
    fn_mask = (y_true == 1) & (y_pred == 0)
    fn_indices = np.where(fn_mask)[0]
    
    print(f"\nMisclassification Summary:")
    print(f"  Total misclassified: {misclassified.sum()} out of {len(y_true)} ({misclassified.sum()/len(y_true)*100:.2f}%)")
    print(f"  False Positives (Ham→Spam): {len(fp_indices)} ({len(fp_indices)/len(y_true)*100:.2f}%)")
    print(f"  False Negatives (Spam→Ham): {len(fn_indices)} ({len(fn_indices)/len(y_true)*100:.2f}%)")
    
    print("\n" + "="*80)
    print(f"FALSE POSITIVES: Ham Emails Incorrectly Classified as Spam")
    print("="*80)
    print("\nThese are legitimate emails that were flagged as spam.")
    print("High FP rate = annoying for users (important emails in spam folder)")
    
    if len(fp_indices) > 0:
        print(f"\nShowing {min(num_examples, len(fp_indices))} examples:")
        for i, idx in enumerate(fp_indices[:num_examples]):
            print(f"\n--- False Positive Example {i+1} (Index: {idx}) ---")
            print(f"Email preview (first 200 chars):")
            print(f"{cleaned_emails[idx][:200]}...")
            
            # Analyze common patterns
            email_text = cleaned_emails[idx].lower()
            spam_keywords = ['free', 'click', 'money', 'offer', 'buy', 'win', 'prize', 'sale']
            found_keywords = [kw for kw in spam_keywords if kw in email_text]
            if found_keywords:
                print(f"Possible reason: Contains spam-like keywords: {found_keywords}")
    else:
        print("\nNo false positives! Perfect precision for ham.")
    
    print("\n" + "="*80)
    print(f"FALSE NEGATIVES: Spam Emails Incorrectly Classified as Ham")
    print("="*80)
    print("\nThese are spam emails that got through the filter.")
    print("High FN rate = spam in inbox (bad user experience)")
    
    if len(fn_indices) > 0:
        print(f"\nShowing {min(num_examples, len(fn_indices))} examples:")
        for i, idx in enumerate(fn_indices[:num_examples]):
            print(f"\n--- False Negative Example {i+1} (Index: {idx}) ---")
            print(f"Email preview (first 200 chars):")
            print(f"{cleaned_emails[idx][:200]}...")
            
            email_text = cleaned_emails[idx].lower()
            if len(email_text.split()) < 10:
                print(f"Possible reason: Very short email ({len(email_text.split())} words) - hard to classify")
            elif 'unsubscribe' in email_text or 'opt out' in email_text:
                print(f"Possible reason: Contains 'unsubscribe' - looks like legitimate newsletter")
    else:
        print("\nNo false negatives! Perfect recall for spam.")
    
    # Provide improvement suggestions
    print("\n" + "="*80)
    print("IMPROVEMENT SUGGESTIONS")
    print("="*80)
    
    print("\nBased on misclassification analysis:")
    
    if len(fp_indices) > 0:
        print("\nTo reduce False Positives (Ham→Spam):")
        print("  - Consider adjusting decision threshold to be more conservative")
        print("  - Some ham emails may contain promotional language")
        print("  - Feature engineering: distinguish between personal and promotional ham")
    
    if len(fn_indices) > 0:
        print("\nTo reduce False Negatives (Spam→Ham):")
        print("  - Consider adjusting decision threshold to be more aggressive")
        print("  - Some spam may be sophisticated (mimicking legitimate email)")
        print("  - Feature engineering: add more sophisticated features")
    
    print("\nGeneral improvements:")
    print("  - Examine specific misclassified examples for patterns")
    print("  - Consider domain-specific features (sender reputation, etc.)")
    print("  - Analyze if certain clusters have higher error rates")
    print("  - Note: Per rubric, stay with Naive Bayes (no other classifiers)")
    
    return fp_indices, fn_indices


**`threshold_analysis`** uses out-of-fold class probabilities (spam probability = column 1), not discrete predictions. It sweeps decision thresholds (e.g. 0.0 to 1.0 in steps of 0.05), recomputes precision, recall, F1, and accuracy at each threshold, and builds a four-panel figure: precision/recall vs threshold, F1/accuracy vs threshold, ROC curve with AUC, and precision-recall curve. It marks the default 0.5 and an optional F1-maximizing threshold and prints brief guidance on the trade-offs of lowering or raising the threshold.

In [ ]:
def threshold_analysis(y_true, y_pred_proba):
    
    # Extract spam probabilities (column 1)
    y_scores = y_pred_proba[:, 1]
    
    print("\nDefault threshold: 0.5")
    print("Analyzing thresholds from 0.0 to 1.0...")
    
    # Test different thresholds
    thresholds = np.arange(0.0, 1.01, 0.05)
    precisions = []
    recalls = []
    f1_scores = []
    accuracies = []
    
    for threshold in thresholds:
        y_pred_thresh = (y_scores >= threshold).astype(int)
        
        # Calculate metrics
        acc = accuracy_score(y_true, y_pred_thresh)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred_thresh, average='binary', zero_division=0
        )
        
        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
    
    fpr, tpr, roc_thresholds = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    print(f"\nROC AUC Score: {roc_auc:.4f}")
    print("(AUC = 1.0 is perfect, 0.5 is random)")
    
    # Find optimal threshold (maximizes F1)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    print(f"\nOptimal threshold (max F1): {optimal_threshold:.2f}")
    print(f"  Precision: {precisions[optimal_idx]:.4f}")
    print(f"  Recall:    {recalls[optimal_idx]:.4f}")
    print(f"  F1-Score:  {f1_scores[optimal_idx]:.4f}")
    print(f"  Accuracy:  {accuracies[optimal_idx]:.4f}")
    
    # Create visualizations
    print("\nCreating threshold analysis visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Decision Threshold Analysis', fontsize=20, fontweight='bold')
    
    # Plot 1: Precision and Recall vs Threshold
    ax1 = axes[0, 0]
    ax1.plot(thresholds, precisions, 'b-', linewidth=2, marker='o', label='Precision')
    ax1.plot(thresholds, recalls, 'r-', linewidth=2, marker='s', label='Recall')
    ax1.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Default (0.5)')
    ax1.axvline(x=optimal_threshold, color='green', linestyle='--', alpha=0.7, 
               label=f'Optimal ({optimal_threshold:.2f})')
    ax1.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Precision and Recall vs Threshold', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-0.05, 1.05)
    ax1.set_ylim(0, 1.05)
    
    # Plot 2: F1-Score and Accuracy vs Threshold
    ax2 = axes[0, 1]
    ax2.plot(thresholds, f1_scores, 'g-', linewidth=2, marker='^', label='F1-Score')
    ax2.plot(thresholds, accuracies, 'm-', linewidth=2, marker='d', label='Accuracy')
    ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Default (0.5)')
    ax2.axvline(x=optimal_threshold, color='green', linestyle='--', alpha=0.7, 
               label=f'Optimal ({optimal_threshold:.2f})')
    ax2.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax2.set_title('F1-Score and Accuracy vs Threshold', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_ylim(0, 1.05)
    
    # Plot 3: ROC Curve
    ax3 = axes[1, 0]
    ax3.plot(fpr, tpr, 'b-', linewidth=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
    ax3.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier (AUC = 0.5)')
    ax3.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax3.set_ylabel('True Positive Rate (Recall)', fontsize=12, fontweight='bold')
    ax3.set_title('ROC Curve', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=11, loc='lower right')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(-0.05, 1.05)
    ax3.set_ylim(-0.05, 1.05)
    
    # Plot 4: Precision-Recall Curve
    ax4 = axes[1, 1]
    # Calculate precision-recall curve
    from sklearn.metrics import precision_recall_curve
    pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
    ax4.plot(pr_recall, pr_precision, 'g-', linewidth=3, label='Precision-Recall Curve')
    ax4.set_xlabel('Recall', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Precision', fontsize=12, fontweight='bold')
    ax4.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(-0.05, 1.05)
    ax4.set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    
    # Interpretation guidance
    print("\n" + "="*80)
    print("THRESHOLD SELECTION GUIDANCE")
    print("="*80)
    print("\nTrade-offs when adjusting threshold:")
    print("\nLower threshold (< 0.5):")
    print("  - Higher recall (catches more spam)")
    print("  - Lower precision (more false positives - ham marked as spam)")
    print("  - Use when: Missing spam is worse than blocking ham")
    print("\nHigher threshold (> 0.5):")
    print("  - Higher precision (fewer false positives)")
    print("  - Lower recall (misses more spam)")
    print("  - Use when: Blocking ham is worse than letting spam through")
    print("\nFor spam filtering:")
    print("  - Typically prioritize recall (catch spam)")
    print("  - But false positives are very annoying to users")
    print("  - Default 0.5 is often a reasonable compromise")
    
    metrics_by_threshold = {
        'thresholds': thresholds,
        'precisions': precisions,
        'recalls': recalls,
        'f1_scores': f1_scores,
        'accuracies': accuracies,
        'optimal_threshold': optimal_threshold,
        'roc_auc': roc_auc,
        'fpr': fpr,
        'tpr': tpr
    }
    
    return fig, metrics_by_threshold


**`create_additional_visualizations`** produces a four-panel performance summary figure. It uses per-fold accuracy from CV (`fold_scores`) and the metrics dictionary from `generate_classification_metrics` (which is based on out-of-fold predictions at the default threshold). Panels show: bar chart of fold accuracies with mean line; per-class precision, recall, and F1 (ham vs spam); pie chart of confusion matrix distribution (TN, FP, FN, TP); and horizontal bar chart of overall accuracy and macro precision, recall, and F1. All values are labeled on the plot for presentation use.

In [ ]:
def create_additional_visualizations(fold_scores, metrics_dict):

    print("\n" + "="*80)
    print("CREATING ADDITIONAL VISUALIZATIONS")
    print("="*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Model Performance Summary', fontsize=20, fontweight='bold')
    
    # Plot 1: Cross-validation fold scores
    ax1 = axes[0, 0]
    folds = range(1, len(fold_scores) + 1)
    ax1.bar(folds, fold_scores, color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax1.axhline(y=np.mean(fold_scores), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {np.mean(fold_scores):.4f}')
    ax1.set_xlabel('Fold Number', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title(f'Cross-Validation Fold Accuracies (Mean±Std: {np.mean(fold_scores):.4f}±{np.std(fold_scores):.4f})',
                 fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(0.8, 1.0)
    ax1.set_xticks(folds)
    
    # Plot 2: Per-class metrics comparison
    ax2 = axes[0, 1]
    metrics = ['Precision', 'Recall', 'F1-Score']
    ham_scores = [metrics_dict['ham_precision'], metrics_dict['ham_recall'], metrics_dict['ham_f1']]
    spam_scores = [metrics_dict['spam_precision'], metrics_dict['spam_recall'], metrics_dict['spam_f1']]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax2.bar(x - width/2, ham_scores, width, label='Ham', color='green', alpha=0.7)
    bars2 = ax2.bar(x + width/2, spam_scores, width, label='Spam', color='red', alpha=0.7)
    
    ax2.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax2.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metrics, fontsize=11)
    ax2.legend(fontsize=11)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(0.8, 1.0)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=9)
    
    # Plot 3: Confusion matrix percentages (pie chart style)
    ax3 = axes[1, 0]
    sizes = [metrics_dict['tn'], metrics_dict['fp'], metrics_dict['fn'], metrics_dict['tp']]
    labels = [f"TN\n{metrics_dict['tn']}", f"FP\n{metrics_dict['fp']}", 
              f"FN\n{metrics_dict['fn']}", f"TP\n{metrics_dict['tp']}"]
    colors = ['lightgreen', 'orange', 'red', 'darkgreen']
    explode = (0.05, 0.05, 0.05, 0.05)
    
    ax3.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.2f%%',
           shadow=True, startangle=45, textprops={'fontsize': 11, 'fontweight': 'bold'})
    ax3.set_title('Confusion Matrix Distribution', fontsize=14, fontweight='bold')
    
    # Plot 4: Key metrics summary bar chart
    ax4 = axes[1, 1]
    summary_metrics = ['Accuracy', 'Precision\n(Macro)', 'Recall\n(Macro)', 'F1-Score\n(Macro)']
    summary_values = [
        metrics_dict['accuracy'],
        metrics_dict['macro_precision'],
        metrics_dict['macro_recall'],
        metrics_dict['macro_f1']
    ]
    
    bars = ax4.barh(summary_metrics, summary_values, color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax4.set_xlabel('Score', fontsize=12, fontweight='bold')
    ax4.set_title('Overall Model Performance Summary', fontsize=14, fontweight='bold')
    ax4.set_xlim(0.8, 1.0)
    ax4.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (bar, value) in enumerate(zip(bars, summary_values)):
        ax4.text(value, i, f' {value:.4f}', va='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    
    print("Additional visualizations created successfully!")
    
    return fig


In [ ]:
# Create confusion matrix (required by rubric)
# Must show all samples (100% of data) with out-of-fold predictions
confusion_fig, cm = create_confusion_matrix_plot(labels, y_pred)


In [ ]:
# Generate classification report
metrics_dict = generate_classification_metrics(labels, y_pred)


In [ ]:
# Analyze misclassifications
# Examine false positives and false negatives to understand errors
fp_indices, fn_indices = analyze_misclassifications(
    labels, y_pred, cleaned_emails, labels, num_examples=10
)


In [ ]:
# Decision threshold analysis (required by rubric)
# Shows how adjusting threshold affects precision, recall, accuracy
# Includes ROC curve and AUC score
threshold_fig, threshold_metrics = threshold_analysis(labels, y_pred_proba)


In [ ]:
# Additional visualizations (required by rubric)
# Summary plots showing overall model performance
summary_fig = create_additional_visualizations(fold_scores, metrics_dict)


In [ ]:
# Final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nFinal Accuracy (All Folds): {metrics_dict['accuracy']:.4f}")
print(f"Mean CV Accuracy: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
print(f"\nHam Performance:")
print(f"  Precision: {metrics_dict['ham_precision']:.4f}")
print(f"  Recall:    {metrics_dict['ham_recall']:.4f}")
print(f"  F1-Score:  {metrics_dict['ham_f1']:.4f}")
print(f"\nSpam Performance:")
print(f"  Precision: {metrics_dict['spam_precision']:.4f}")
print(f"  Recall:    {metrics_dict['spam_recall']:.4f}")
print(f"  F1-Score:  {metrics_dict['spam_f1']:.4f}")
print(f"\nROC AUC: {threshold_metrics['roc_auc']:.4f}")
